#### Sample script for reading in TC-DROPS data
Brian Tang

<i> 6 November 2025 </i>

In [ ]:
#Libraries

import re
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from metpy.calc import dewpoint_from_relative_humidity
from metpy.plots import SkewT
from metpy.units import units
import seaborn as sns
sns.set_context("paper",font_scale=1.0)

%matplotlib inline

In [ ]:
#open TC-DROPS dataset (we'll open the data on height levels)
ds_tcdrops = xr.open_dataset('/network/rit/lab/tcdynasty/tang/tcdrops/all_storms_height_dropsonde_dataset_ver1.2_updated.nc')

#print dataset info
ds_tcdrops

In [ ]:
#Specify storm and mission(s) of interest
atcfid = 'AL062014' #ATCF identifier
tcdropsid = 'al_edouard2014' #TCDROPS identifier <basin>_<name><year>
storm = re.search(r'_(\D+)\d+', tcdropsid).group(1) #extract just the storm name

#print all available missions for storm
stormname = np.squeeze(ds_tcdrops['storm_name'].values).astype(str)
ind = np.nonzero(stormname == tcdropsid)[0]
flightids = np.squeeze(ds_tcdrops['flight_id'].values).astype(str)
print(np.unique(flightids[ind]))

#specify missions of interest that you want to extract dropsonde data for
flightid = ['20140914I1', '20140914_hs3_gh'] #can list one mission; group missions together to examine combined dropsondes

In [ ]:
#Open IBTracs to get TC track observations
ds_ibtracs = xr.open_dataset('/network/rit/lab/tcdynasty/tang/ibtracs/IBTrACS.since1980.v04r01.nc')

#determine index of TC
atcf = np.squeeze(ds_ibtracs['usa_atcf_id'][:,0].values).astype(str)
atcfind = np.nonzero(atcf == atcfid)[0][0]
print(atcfind)

#get track information of TC
ds_ibtracs_tc = ds_ibtracs[['usa_lat','usa_lon']].isel(storm = atcfind)

#convert to dataframe
df_ibtracs_tc = ds_ibtracs_tc.to_dataframe().dropna(how='all').reset_index()
df_ibtracs_tc = df_ibtracs_tc.drop(columns=['date_time','usa_lat','usa_lon'])
df_ibtracs_tc['time'] = df_ibtracs_tc['time'].dt.round('min')

#resample to every minute and apply cubic interpolation
df_ibtracs_tc = df_ibtracs_tc.set_index('time').resample('1s').interpolate(method='cubic')
df_ibtracs_tc = df_ibtracs_tc.reset_index()

# calculate storm motion
REarth = 6371.8E3
time_array = (df_ibtracs_tc['time']-df_ibtracs_tc['time'].iloc[0])/np.timedelta64(1,'s')
# zonal motion = d(lon)/dt * cos(lat) * Radius of Earth
df_ibtracs_tc['storm_motion_u'] = np.gradient(np.radians(df_ibtracs_tc.lon.values),time_array.values)*REarth*np.cos(np.radians(df_ibtracs_tc.lat.values))
# meridional motion = d(lat)/dt * Radius of Earth
df_ibtracs_tc['storm_motion_v'] = np.gradient(np.radians(df_ibtracs_tc.lat.values),time_array.values)*REarth
df_ibtracs_tc

In [ ]:
#Extract dropsonde data for specified mission(s) of interest
#subset mission(s) of interest
ind = []
for i in range(len(flightid)):
    ind.append(np.nonzero(flightids == flightid[i])[0])
ind = np.hstack(ind)

#get the launch time for each drop
drops = ds_tcdrops[['year','month','day','hour','minute','second','latitude_initial','longitude_initial',\
                       'lat_dist_initial','lon_dist_initial','storm_name','flight_id']].isel(ndrop = ind)
df_mission = drops.to_dataframe()
df_mission['storm_name'] = df_mission['storm_name'].str.decode("utf-8")
df_mission['flight_id'] = df_mission['flight_id'].str.decode("utf-8")

#convert launch date/time information to datetime
df_mission['launchtime'] = df_mission.apply(lambda row: datetime(int(row['year']),int(row['month']),int(row['day']),\
                                        int(row['hour']),int(row['minute']),int(row['second'])) ,axis=1)

#get all drops for mission
ds_tcdrop = ds_tcdrops[['time_elapsed','longitude','latitude','pressure','height','temperature','wvmr','rh','uwind','vwind','theta_e']].isel(ndrop = ind)

#convert to dataframe and clean up
df_tcdrop = ds_tcdrop.to_dataframe().dropna(how='all').reset_index()

#calculate time of observation
df_tcdrop = pd.merge(df_tcdrop,df_mission['launchtime'],on='ndrop')
df_tcdrop['time'] = df_tcdrop['launchtime'] + pd.to_timedelta(df_tcdrop['time_elapsed'], unit='s')
df_tcdrop = df_tcdrop.drop(columns=['time_elapsed','nlevels','launchtime'])

#merge with resampled IBTracs data
df_tcdrop['time'] = df_tcdrop['time'].dt.round('s') #round time in TC-DROPS to nearest second
df_tcdrop = pd.merge(df_tcdrop,df_ibtracs_tc,on='time')

#rename TC position columns to avoid confusion
df_tcdrop = df_tcdrop.rename(columns={'lat': 'tclat', 'lon': 'tclon'})

df_tcdrop

In [ ]:
#Perform calculations to transform variables into cylindrical grid and storm-relative frame of reference
#calculate x and y distance relative to center, along with radius and azimuth
tcphi = df_tcdrop.tclat*np.pi/180
tclam = df_tcdrop.tclon*np.pi/180
phi = df_tcdrop.latitude*np.pi/180
lam = df_tcdrop.longitude*np.pi/180

#transformation variables to account for sphericity of lat/lon grid, following Ahern and Cowan (2018)
a = np.clip(np.sin(tcphi)*np.sin(phi)+np.cos(tcphi)*np.cos(phi)*np.cos(phi-tcphi),-1,1)
ctrangle = np.maximum(np.arccos(a),1E-3)
df_tcdrop['x_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*np.cos(phi)*np.sin(lam-tclam)
df_tcdrop['y_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*(np.cos(tcphi)*np.sin(phi)-\
                                np.sin(tcphi)*np.cos(phi)*np.cos(lam-tclam))
df_tcdrop['radius'] = (df_tcdrop.x_dist**2.0+df_tcdrop.y_dist**2.0)**0.5
theta_xy = np.arctan2(df_tcdrop.y_dist,df_tcdrop.x_dist)
df_tcdrop['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#Calculate storm-relative u, v, radial, and tangential wind, following Ahern and Cowan (2018)
df_tcdrop['u_wind_relative']=df_tcdrop.uwind-df_tcdrop.storm_motion_u
df_tcdrop['v_wind_relative']=df_tcdrop.vwind-df_tcdrop.storm_motion_v

eps = 1.E-3
wspd = np.sqrt(df_tcdrop.u_wind_relative**2+df_tcdrop.v_wind_relative**2)
lamprime = lam + eps*df_tcdrop.u_wind_relative/(wspd*np.cos(phi))
phiprime = phi + eps*df_tcdrop.v_wind_relative/wspd

b = np.clip(np.sin(tcphi)*np.sin(phiprime)+np.cos(tcphi)*np.cos(phiprime)*np.cos(phiprime-tcphi),-1,1)
ctrangleprime = np.maximum(np.arccos(b),1E-3)
xprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*np.cos(phiprime)*np.sin(lamprime-tclam)
yprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*(np.cos(tcphi)*np.sin(phiprime)-\
                        np.sin(tcphi)*np.cos(phiprime)*np.cos(lamprime-tclam))
alpha = np.arctan2(yprime-df_tcdrop.y_dist,xprime-df_tcdrop.x_dist)
alpha = np.where(alpha>=0,alpha,alpha+2*np.pi)

df_tcdrop['radwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha)
df_tcdrop['tanwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha+np.pi/2)

In [ ]:
#Get rid of NaNs and sort
df_tcdrop = df_tcdrop.set_index('time').groupby('ndrop', group_keys=True).apply(lambda group : group.interpolate(method='time',limit_area='inside'))
df_tcdrop = df_tcdrop.drop(columns=['ndrop'])
df_tcdrop = df_tcdrop.dropna().sort_values(['ndrop','time'],ascending=[True,False]).reset_index()

df_tcdrop

#### Examples diagnostics and visualizations

In [ ]:
#Plot dropsonde points in TC-relative frame of reference
#group into drops from low-level and high-level aircraft
pltlim = 400
df_tcdrop_release = df_tcdrop.groupby('ndrop').tail(1)
df_tcdrop_release_hi = df_tcdrop_release[df_tcdrop_release['pressure']<500] #low-level recon drops
df_tcdrop_release_lo = df_tcdrop_release[df_tcdrop_release['pressure']>=500] #high-level recon drops

#plot release points (note: only plots points within axes range)
fig = plt.figure(figsize=(8,8))
plt.plot(df_tcdrop_release_hi['x_dist']/1E3,df_tcdrop_release_hi['y_dist']/1E3,'x',color='red',ms=7,mew=1.5)
plt.plot(df_tcdrop_release_lo['x_dist']/1E3,df_tcdrop_release_lo['y_dist']/1E3,'+',color='blue',ms=8,mew=1.5)
for j in range(len(df_tcdrop_release)):
    plt.text(df_tcdrop_release['x_dist'].iloc[j]/1E3,df_tcdrop_release['y_dist'].iloc[j]/1E3,df_tcdrop_release['ndrop'].iloc[j],clip_on=True)
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

#range rings
crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
#Plot skew-T and ventilation profile
#select dropsonde to plot
sid = 78 #isolate 

fig = plt.figure(figsize=(8,8))
skew = SkewT(fig, rotation=35, aspect=90)

df_sonde = df_tcdrop[(df_tcdrop['ndrop'] == sid)].sort_values(by=['pressure'])
p = df_sonde['pressure'].values * units.hPa
T = df_sonde['temperature'].values * units.degC
u = df_sonde['uwind'].values * units('m/s')
v = df_sonde['vwind'].values * units('m/s')
Td = dewpoint_from_relative_humidity(df_sonde['temperature'].values * units.degC, df_sonde['rh'].values * units.percent)
skew.plot(p, T, lw=2.5, color='tab:red')
skew.plot(p, Td,lw=2.5, color='tab:green')
skew.plot_barbs(p[::10], u[::10], v[::10], length=6)

skew.ax.set_ylim(1012, 200)
skew.ax.set_xlim(-25, 30)
skew.ax.set_xlabel('Temperature, dewpoint (C)')
skew.ax.set_ylabel('Pressure (hPa)')

skew.plot_dry_adiabats(lw=0.5)
skew.plot_moist_adiabats(lw=0.5)
skew.plot_mixing_lines(lw=0.5)

plt.show()

In [ ]:
#Make a plot of values of some variable averaged over a given layer. Use theta_e as an example.
#determine change of pressure between observations to use as weights in layer average
df_tcdrop['deltap'] = ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(1))/2) - \
                      ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(-1))/2)
df_tcdrop['deltap'] = df_tcdrop['deltap'].fillna(0)
df_tcdrop['deltap'] = df_tcdrop['deltap'].mask(df_tcdrop['deltap'] < 0, 0)

#specify layer of interest that you want to average over
df_lay = df_tcdrop.loc[(df_tcdrop['pressure'] >= 900) & (df_tcdrop['pressure'] <= 1000)].groupby('ndrop')

#calculate layer average (add variables to list that you want to average)
df_lay_avg = df_lay.apply(lambda x: pd.Series(np.average(x[['x_dist','y_dist','theta_e']], \
                            weights=x['deltap'], axis=0), ['x_dist','y_dist','theta_e'])).dropna()

#determine radius of azimuth of average locations
theta_xy = np.arctan2(df_lay_avg.y_dist,df_lay_avg.x_dist)
df_lay_avg['radius'] = (df_lay_avg.x_dist**2.0+df_lay_avg.y_dist**2.0)**0.5
df_lay_avg['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#make plot
pltlim = 400
fig = plt.figure(figsize=(10,8))
h = plt.scatter(df_lay_avg['x_dist']/1E3,df_lay_avg['y_dist']/1E3,c=df_lay_avg['theta_e'],s=150,edgecolors='black',cmap=plt.cm.Spectral_r,vmin=340,vmax=370)
cb = plt.colorbar(h, shrink=0.75)
cb.set_label(r'$\theta_e\;(K)$')
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from pyproj import Geod
from datetime import timedelta

geod = Geod(ellps='WGS84')

def label_RI_RW(bt_df, hours_between=24, wind_col='vmax_kt', time_col='time'):
    bt = bt_df.sort_values(time_col).set_index(time_col)
    step = int(hours_between / (bt.index.to_series().diff().median().total_seconds() / 3600.0))
    bt['vmax_next'] = bt[wind_col].shift(-step)
    bt['dv'] = bt['vmax_next'] - bt[wind_col]
    RI = bt[bt['dv'] >= 30]
    RW = bt[bt['dv'] <= -30]
    return RI.index.tolist(), RW.index.tolist(), bt

def to_storm_relative(storm_lat, storm_lon, pts_df):
    azs, dists = [], []
    for lat, lon in zip(pts_df['lat'].values, pts_df['lon'].values):
        az1, az2, dist = geod.inv(storm_lon, storm_lat, lon, lat)
        azs.append((az1 + 360) % 360)
        dists.append(dist/1000.0)
    pts = pts_df.copy()
    pts['azimuth_deg'] = azs
    pts['distance_km'] = dists
    return pts

def compute_event_metrics(onset_time, bt_df, flashes_df, outer_rmin=100, outer_rmax=500, pre_hours=24, post_hours=24):
    # find storm position at onset_time (interpolate if necessary)
    storm_row = bt_df.loc[onset_time]
    s_lat, s_lon = storm_row['lat'], storm_row['lon']
    # window
    t0 = pd.to_datetime(onset_time)
    win = (flashes_df['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (flashes_df['time'] < t0 + pd.Timedelta(hours=post_hours))
    fl_win = flashes_df[win].copy()
    fl_rel = to_storm_relative(s_lat, s_lon, fl_win)
    # outer flashes pre/post
    pre_mask = (fl_rel['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (fl_rel['time'] < t0)
    post_mask = (fl_rel['time'] >= t0) & (fl_rel['time'] < t0 + pd.Timedelta(hours=post_hours))
    outer_mask = (fl_rel['distance_km'] >= outer_rmin) & (fl_rel['distance_km'] <= outer_rmax)
    outer_pre = fl_rel[pre_mask & outer_mask]
    outer_post = fl_rel[post_mask & outer_mask]
    # counts and rates
    metrics = {
        'outer_pre_count': len(outer_pre),
        'outer_post_count': len(outer_post),
        'outer_pre_rate_per_hr': len(outer_pre) / pre_hours,
        'outer_post_rate_per_hr': len(outer_post) / post_hours,
    }
    # area km^2
    import math
    area = math.pi * (outer_rmax**2 - outer_rmin**2)
    metrics['outer_pre_density_per_km2_hr'] = metrics['outer_pre_rate_per_hr'] / area
    metrics['outer_post_density_per_km2_hr'] = metrics['outer_post_rate_per_hr'] / area
    return metrics


In [ ]:
###########################################          My code here

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# correct working directory
os.chdir("/nfs/tcdynasty/esmith/SATELLITE_/edouard_4km_ir")

# open the image
sat_file = "2014al06_4kmirimg_201409150415.gif"
img = Image.open(sat_file).convert("RGBA")

# check size
width, height = img.size
print("Image size:", width, height)

# display
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.show()


In [ ]:
import numpy as np

# --- Display the image first ---
fig, ax = plt.subplots(figsize=(8,8))
ax.imshow(img)
ax.axis("off")  # optional

# --- Map dropsonde coordinates to image ---
# Example: df_tcdrop_release already has 'x_dist' and 'y_dist' in km
# You need to scale them to image pixels. We'll assume:
#  - image_center corresponds to (x=0, y=0) km
#  - the max range of your plot corresponds roughly to image size

# Define limits (in km) to match your previous plot
pltlim = 400  # km
x = df_tcdrop_release['x_dist'] / 1e3  # convert m -> km if needed
y = df_tcdrop_release['y_dist'] / 1e3

# Image dimensions
img_width, img_height = img.size

# Convert km -> pixel
# Map x from [-pltlim, pltlim] -> [0, img_width]
x_pix = (x + pltlim) / (2*pltlim) * img_width
# Map y from [-pltlim, pltlim] -> [img_height, 0] (invert y-axis for images)
y_pix = img_height - (y + pltlim) / (2*pltlim) * img_height

# --- Plot dropsonde points ---
# High-level dropsonde (pressure < 500 hPa)
hi_idx = df_tcdrop_release['pressure'] < 500
ax.scatter(x_pix[hi_idx], y_pix[hi_idx], color='red', s=50, marker='x', label='High-level drops')

# Low-level dropsonde (pressure >= 500 hPa)
lo_idx = df_tcdrop_release['pressure'] >= 500
ax.scatter(x_pix[lo_idx], y_pix[lo_idx], color='blue', s=50, marker='+', label='Low-level drops')

# Optional: annotate dropsonde IDs
for j in range(len(df_tcdrop_release)):
    ax.text(x_pix[j], y_pix[j], str(df_tcdrop_release['ndrop'].iloc[j]),
            color='white', fontsize=8, ha='center', va='center')

ax.legend()
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# --- Open image ---
sat_file = "2014al06_4kmirimg_201409150415.gif"
img = Image.open(sat_file).convert("RGBA")
img_width, img_height = img.size
print("Image size:", img_width, img_height)

# --- Prepare dropsonde data ---
df_release = df_tcdrop.groupby('ndrop').tail(1).reset_index(drop=True)
df_release_hi = df_release[df_release['pressure'] < 500].reset_index(drop=True)
df_release_lo = df_release[df_release['pressure'] >= 500].reset_index(drop=True)

# Map TC-relative x,y (km) to image pixels
pltlim = 400  # km from center to edge
x_center = img_width // 2
y_center = img_height // 2

def km_to_pixel(x_km, y_km, img_w, img_h, pltlim):
    scale_x = (img_w // 2) / pltlim
    scale_y = (img_h // 2) / pltlim
    x_pix = x_center + (x_km * scale_x)
    y_pix = y_center - (y_km * scale_y)  # flip y-axis
    return x_pix, y_pix

x_pix_hi, y_pix_hi = km_to_pixel(df_release_hi['x_dist']/1e3, df_release_hi['y_dist']/1e3, img_width, img_height, pltlim)
x_pix_lo, y_pix_lo = km_to_pixel(df_release_lo['x_dist']/1e3, df_release_lo['y_dist']/1e3, img_width, img_height, pltlim)

# --- Plot image and dropsondes ---
fig, ax = plt.subplots(figsize=(img_width/100, img_height/100))
ax.imshow(img, extent=[0, img_width, 0, img_height])  # use full image
ax.scatter(x_pix_hi, y_pix_hi, c='red', s=30, marker='x', label='High-level dropsonde')
ax.scatter(x_pix_lo, y_pix_lo, c='blue', s=30, marker='+', label='Low-level dropsonde')

# Annotate dropsonde IDs
for df_sub, x_pix_arr, y_pix_arr in zip([df_release_hi, df_release_lo], [x_pix_hi, x_pix_lo], [y_pix_hi, y_pix_lo]):
    for i, txt in enumerate(df_sub['ndrop']):
        ax.text(x_pix_arr[i], y_pix_arr[i], str(txt), color='white', fontsize=6, ha='center', va='center')

ax.set_xlim(0, img_width)
ax.set_ylim(img_height, 0)  # flip y-axis to match image
ax.axis('off')
ax.legend(fontsize=6)
plt.show()


In [ ]:
import pandas as pd
from datetime import datetime

# --- Extract satellite time from filename ---
# Example filename: "2014al06_4kmirimg_201409150415.gif"
sat_time_str = sat_file.split('_')[-1].split('.')[0]  # '201409150415'
sat_time = datetime.strptime(sat_time_str, "%Y%m%d%H%M")

# --- Get dropsonde release times ---
df_release = df_tcdrop.groupby('ndrop').tail(1).reset_index(drop=True)
dropsonde_times = df_release['time'].dt.strftime("%Y-%m-%d %H:%M:%S")

# --- Create a table ---
table_df = pd.DataFrame({
    'Dropsonde ID': df_release['ndrop'].values,
    'Dropsonde Time': dropsonde_times,
    'Satellite Image Time': [sat_time.strftime("%Y-%m-%d %H:%M:%S")] * len(df_release)
})

# Display table
table_df


In [ ]:
import os
from datetime import datetime
import pandas as pd

# --- Set working directory to SATELLITE_ folder ---
os.chdir("/nfs/tcdynasty/esmith/SATELLITE_")

# --- Folder containing satellite images ---
sat_folder = "edouard_4km_ir"

# --- List all .gif files in the folder ---
sat_files = [f for f in os.listdir(sat_folder) if f.endswith(".gif")]

# --- Extract datetime from each satellite filename ---
sat_times = []
for f in sat_files:
    # Example filename: 2014al06_4kmirimg_201409150415.gif
    dt_str = f.split("_")[-1].split(".gif")[0]  # "201409150415"
    sat_times.append(datetime.strptime(dt_str, "%Y%m%d%H%M"))

# --- Dropsonde times ---
drop_times = df_tcdrop['time'].sort_values().unique()

# --- Create a DataFrame for easy viewing ---
df_times = pd.DataFrame({
    "Dropsonde Times": pd.Series(drop_times),
    "Satellite Times": pd.Series(sat_times)
})

# --- Show table ---
df_times


In [ ]:
# Ensure both arrays are sorted
drop_times_sorted = pd.Series(df_tcdrop['time'].sort_values()).reset_index(drop=True)
sat_times_sorted = pd.Series(sorted(sat_times)).reset_index(drop=True)

# Create DataFrames
df_drop = pd.DataFrame({"Dropsonde Times": drop_times_sorted})
df_sat = pd.DataFrame({"Satellite Times": sat_times_sorted})

# Merge dropsonde times with the nearest satellite time
df_aligned = pd.merge_asof(df_drop, df_sat, left_on="Dropsonde Times", right_on="Satellite Times", direction="nearest")

# Optional: limit max time difference (e.g., 30 min) to avoid really far matches
max_diff = pd.Timedelta("30min")
mask = (df_aligned['Satellite Times'] - df_aligned['Dropsonde Times']).abs() <= max_diff
df_aligned = df_aligned[mask]

df_aligned



In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pandas as pd
import numpy as np

# --- Folder and files ---
sat_folder = "/nfs/tcdynasty/esmith/SATELLITE_/edouard_4km_ir"
sat_files = sorted([f for f in os.listdir(sat_folder) if f.endswith(".gif")])

# --- Extract satellite times from filenames ---
# Example filename: 2014al06_4kmirimg_201409150415.gif
sat_times = [pd.to_datetime(f.split("_")[-1].split(".")[0], format="%Y%m%d%H%M") for f in sat_files]

# --- Combine into DataFrame for easier access ---
df_sat = pd.DataFrame({"filename": sat_files, "time": sat_times})

# --- Dropsonde DataFrame: df_tcdrop (already loaded) ---
# Ensure 'time', 'x_dist', 'y_dist' exist
# We'll convert x_dist/y_dist to approximate pixel coordinates for plotting

# Image size (assuming all images are the same)
img = Image.open(os.path.join(sat_folder, sat_files[0])).convert("RGBA")
img_width, img_height = img.size

# Define plotting area in km for conversion
pltlim = 400  # as used in previous plots
km_to_pix_x = img_width / (2*pltlim)
km_to_pix_y = img_height / (2*pltlim)

# --- Function to get dropsondes near a given time ---
def get_dropsondes_for_image(image_time, df_tcdrop, time_window=pd.Timedelta("2h")):
    mask = (df_tcdrop['time'] >= image_time - time_window) & (df_tcdrop['time'] <= image_time + time_window)
    return df_tcdrop[mask]

# --- Create animation ---
fig, ax = plt.subplots(figsize=(6,6))
ax.axis("off")

def update(frame):
    ax.clear()
    ax.axis("off")
    
    # Load satellite image
    sat_path = os.path.join(sat_folder, df_sat['filename'].iloc[frame])
    img = Image.open(sat_path).convert("RGBA")
    ax.imshow(img)
    
    # Get dropsondes within 2 hours
    image_time = df_sat['time'].iloc[frame]
    df_drops = get_dropsondes_for_image(image_time, df_tcdrop)
    
    if len(df_drops) > 0:
        # Convert x_dist/y_dist to pixel coordinates
        x_pix = img_width/2 + df_drops['x_dist']/1e3*km_to_pix_x
        y_pix = img_height/2 - df_drops['y_dist']/1e3*km_to_pix_y  # flip y-axis
        
        # Low-level drops
        low_mask = df_drops['pressure'] >= 500
        ax.scatter(x_pix[low_mask], y_pix[low_mask], c='blue', s=30, label='Low-level')
        
        # High-level drops
        high_mask = df_drops['pressure'] < 500
        ax.scatter(x_pix[high_mask], y_pix[high_mask], c='red', s=30, marker='x', label='High-level')
    
    # Optional: title with time
    ax.set_title(str(image_time))

# --- Animation ---
anim = FuncAnimation(fig, update, frames=len(df_sat), interval=200)

# --- Save as MP4 ---
writer = FFMpegWriter(fps=5)
anim.save("edouard_dropsonde.mp4", writer=writer)

plt.close(fig)


In [ ]:
# --- Check number of dropsondes ---
print("Total dropsondes in df_tcdrop:", len(df_tcdrop))

# --- Print first few rows ---
print(df_tcdrop[['time', 'pressure', 'x_dist', 'y_dist']].head(10))

# --- Print unique drop IDs and dates ---
if 'ndrop' in df_tcdrop.columns:
    print("Unique dropsonde IDs:", df_tcdrop['ndrop'].unique())
else:
    print("No 'ndrop' column, just printing times:")
    print(df_tcdrop['time'].head(20))

# --- Check time range ---
print("Dropsonde time range:", df_tcdrop['time'].min(), "to", df_tcdrop['time'].max())

# --- Optional: count dropsondes per day ---
df_tcdrop['date'] = df_tcdrop['time'].dt.date
print(df_tcdrop.groupby('date').size())


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pandas as pd
import numpy as np

# --- Paths ---
sat_folder = "/nfs/tcdynasty/esmith/SATELLITE_/edouard_4km_ir"  # folder containing satellite images
output_mp4 = "edouard_dropsonde_animation.mp4"

# --- Time window around each satellite image to include dropsondes (in hours) ---
time_window_hours = 3

# --- Load satellite images ---
sat_files = sorted([f for f in os.listdir(sat_folder) if f.endswith(".gif")])

# --- Extract datetime from satellite filenames ---
sat_times = []
for f in sat_files:
    # Assuming filename format like '2014al06_4kmirimg_YYYYMMDDHHMM.gif'
    dt_str = f.split('_')[-1].replace('.gif', '')  # 'YYYYMMDDHHMM'
    dt = pd.to_datetime(dt_str, format='%Y%m%d%H%M')
    sat_times.append(dt)

# Create DataFrame for convenience
df_sat = pd.DataFrame({'filename': sat_files, 'time': sat_times})

# --- Assume df_tcdrop exists with columns ['time', 'x_dist', 'y_dist'] in meters ---
# Convert x_dist and y_dist from meters to km for plotting if needed
df_tcdrop['x_km'] = df_tcdrop['x_dist'] / 1000
df_tcdrop['y_km'] = df_tcdrop['y_dist'] / 1000

# --- Initialize figure ---
fig, ax = plt.subplots(figsize=(8, 8))

# Optionally, set axis limits if your dropsonde data is within a known range
pltlim = 400  # km
ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)')
ax.set_ylabel('Meridional distance from center (km)')

# --- Function to update each frame ---
def update(frame):
    ax.clear()
    
    # Current satellite image and time
    sat_file = os.path.join(sat_folder, df_sat['filename'].iloc[frame])
    sat_time = df_sat['time'].iloc[frame]
    
    # Open and display satellite image
    img = Image.open(sat_file).convert("RGBA")
    ax.imshow(img, extent=(-pltlim, pltlim, -pltlim, pltlim))  # scale to axis limits

    # Filter dropsondes within ±time_window_hours of satellite time
    time_delta = pd.Timedelta(hours=time_window_hours)
    df_window = df_tcdrop[(df_tcdrop['time'] >= sat_time - time_delta) &
                          (df_tcdrop['time'] <= sat_time + time_delta)]
    
    # Only plot if there are dropsondes
    if not df_window.empty:
        # Optional: separate low/high drops
        df_hi = df_window[df_window['pressure'] < 500]
        df_lo = df_window[df_window['pressure'] >= 500]

        ax.scatter(df_hi['x_km'], df_hi['y_km'], c='red', marker='x', s=50, label='High-level drops')
        ax.scatter(df_lo['x_km'], df_lo['y_km'], c='blue', marker='+', s=50, label='Low-level drops')

        # Optionally annotate dropsonde IDs
        for i, txt in enumerate(df_window.index):
            ax.text(df_window['x_km'].iloc[i], df_window['y_km'].iloc[i],
                    str(txt), color='white', fontsize=6, ha='center', va='center')

        ax.legend(loc='upper right', fontsize=8)

    ax.set_title(f'Satellite: {sat_time.strftime("%Y-%m-%d %H:%M")} UTC', fontsize=10)
    ax.set_xlim(-pltlim, pltlim)
    ax.set_ylim(-pltlim, pltlim)

# --- Create animation ---
anim = FuncAnimation(fig, update, frames=len(df_sat), interval=500)  # 500ms between frames

# --- Save as MP4 ---
writer = FFMpegWriter(fps=2)  # 2 frames per second
anim.save(output_mp4, writer=writer)

print("Animation saved to", output_mp4)


In [ ]:
#-------------------------------------------------------------------------------------------------------------------------

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# --- Paths ---
sat_folder = "/nfs/tcdynasty/esmith/SATELLITE_/edouard_4km_ir"

# --- Pick which satellite frame you want ---
frame_idx = 160  # choose the index of the frame you want to look at
time_window_hours = 3
pltlim = 400  # km

# --- Load satellite files and times ---
sat_files = sorted([f for f in os.listdir(sat_folder) if f.endswith(".gif")])
sat_times = [pd.to_datetime(f.split('_')[-1].replace('.gif',''), format='%Y%m%d%H%M') 
             for f in sat_files]
df_sat = pd.DataFrame({'filename': sat_files, 'time': sat_times})

# --- Filter dropsondes ---
df_tcdrop['x_km'] = df_tcdrop['x_dist'] / 1000
df_tcdrop['y_km'] = df_tcdrop['y_dist'] / 1000

# --- Select the satellite frame ---
sat_file = os.path.join(sat_folder, df_sat['filename'].iloc[frame_idx])
sat_time = df_sat['time'].iloc[frame_idx]

time_delta = pd.Timedelta(hours=time_window_hours)
df_window = df_tcdrop[(df_tcdrop['time'] >= sat_time - time_delta) &
                      (df_tcdrop['time'] <= sat_time + time_delta)]

# --- Plot ---
fig, ax = plt.subplots(figsize=(8,8))
img = Image.open(sat_file).convert("RGBA")
ax.imshow(img, extent=(-pltlim, pltlim, -pltlim, pltlim))

# Plot dropsondes if any
if not df_window.empty:
    df_hi = df_window[df_window['pressure'] < 500]
    df_lo = df_window[df_window['pressure'] >= 500]

    ax.scatter(df_hi['x_km'], df_hi['y_km'], c='red', marker='x', s=50, label='High-level drops')
    ax.scatter(df_lo['x_km'], df_lo['y_km'], c='blue', marker='+', s=50, label='Low-level drops')
    ax.legend(loc='upper right', fontsize=8)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from center (km)')
ax.set_ylabel('Meridional distance from center (km)')
ax.set_title(f'Satellite: {sat_time.strftime("%Y-%m-%d %H:%M")} UTC', fontsize=10)
plt.show()


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# --- Paths ---
sat_folder = "/nfs/tcdynasty/esmith/SATELLITE_/edouard_4km_ir"

# --- Pick which satellite frame you want ---
frame_idx = 160  # index of the frame to look at
time_window_hours = 3
pltlim = 400  # km

# --- Load satellite files and times ---
sat_files = sorted([f for f in os.listdir(sat_folder) if f.endswith(".gif")])
sat_times = [
    pd.to_datetime(f.split('_')[-1].replace('.gif',''), format='%Y%m%d%H%M') 
    for f in sat_files
]
df_sat = pd.DataFrame({'filename': sat_files, 'time': sat_times})

# --- Convert dropsonde distances to km ---
df_tcdrop['x_km'] = df_tcdrop['x_dist'] / 1000
df_tcdrop['y_km'] = df_tcdrop['y_dist'] / 1000

# --- Apply storm center offsets ---
x_offset = 85  # km (storm center is 100 km east)
y_offset = -50    # km (storm center is 50 km south)
df_tcdrop['x_km_centered'] = df_tcdrop['x_km'] - x_offset
df_tcdrop['y_km_centered'] = df_tcdrop['y_km'] - y_offset

# --- Select the satellite frame ---
sat_file = os.path.join(sat_folder, df_sat['filename'].iloc[frame_idx])
sat_time = df_sat['time'].iloc[frame_idx]

# Filter dropsondes within ±time_window_hours
time_delta = pd.Timedelta(hours=time_window_hours)
df_window = df_tcdrop[
    (df_tcdrop['time'] >= sat_time - time_delta) &
    (df_tcdrop['time'] <= sat_time + time_delta)
]

# --- Plot ---
fig, ax = plt.subplots(figsize=(8,8))

# Display satellite image
img = Image.open(sat_file).convert("RGBA")
ax.imshow(img, extent=(-pltlim, pltlim, -pltlim, pltlim))

# Plot dropsondes if any
if not df_window.empty:
    df_hi = df_window[df_window['pressure'] < 500]
    df_lo = df_window[df_window['pressure'] >= 500]

    ax.scatter(df_hi['x_km_centered'], df_hi['y_km_centered'], 
               c='red', marker='x', s=50, label='High-level drops')
    ax.scatter(df_lo['x_km_centered'], df_lo['y_km_centered'], 
               c='blue', marker='+', s=50, label='Low-level drops')
    ax.legend(loc='upper right', fontsize=8)

ax.set_xlim(-pltlim, pltlim)
ax.set_ylim(-pltlim, pltlim)
ax.set_xlabel('Zonal distance from storm center (km)')
ax.set_ylabel('Meridional distance from storm center (km)')
ax.set_title(f'Satellite: {sat_time.strftime("%Y-%m-%d %H:%M")} UTC', fontsize=10)
plt.show()
